In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [2]:
MODEL_NAME    = "bert-base-uncased"
MAX_LENGTH    = 128
BATCH_SIZE    = 32
NUM_EPOCHS    = 1       # reduced from 3
TRAIN_SAMPLES = 10000  # reduced from 120,000
TEST_SAMPLES  = 1000   # reduced from 7,600
OUTPUT_DIR    = "./news_classifier"

ID2LABEL = {0: "World/Politics", 1: "Sports", 2: "Business", 3: "Technology"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

In [3]:
dataset = load_dataset("ag_news")

# Use small subset for fast training
small_train = dataset["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
small_test  = dataset["test"].shuffle(seed=42).select(range(TEST_SAMPLES))

print(f"Train samples : {len(small_train)}")
print(f"Test samples  : {len(small_test)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train samples : 10000
Test samples  : 1000


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

small_train = small_train.map(tokenize, batched=True)
small_test  = small_test.map(tokenize, batched=True)

small_train = small_train.rename_column("label", "labels")
small_test  = small_test.rename_column("label", "labels")

small_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
small_test.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done.")

Tokenization done.


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# View architecture
print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy" : accuracy_score(labels, preds),
        "f1_macro" : f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,              # ← was warmup_ratio
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    processing_class=tokenizer,    # ← was tokenizer=
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Training started…")
trainer.train()
print("Done!")

Training started…


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.283603,0.297154,0.904000,0.903870


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Done!


In [7]:
predictions = trainer.predict(small_test)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"F1 Macro : {f1_score(labels, preds, average='macro'):.4f}\n")
print(classification_report(labels, preds, target_names=list(ID2LABEL.values())))

Accuracy : 0.9040
F1 Macro : 0.9039

                precision    recall  f1-score   support

World/Politics       0.97      0.85      0.91       266
        Sports       0.95      0.99      0.97       246
      Business       0.86      0.86      0.86       246
    Technology       0.83      0.93      0.88       242

      accuracy                           0.90      1000
     macro avg       0.91      0.91      0.90      1000
  weighted avg       0.91      0.90      0.90      1000



In [12]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./news_classifier/tokenizer_config.json', './news_classifier/tokenizer.json')

In [13]:
clf = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

print(clf("Apple unveiled its M4 chip at WWDC with 40% faster neural engine speeds."))
print(clf("Manchester City won the Premier League title on the final day."))
print(clf("Modi Government Won Bengal Elections."))
print(clf("Tesla reported record revenue of $26.2 billion, beating expectations."))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'Technology', 'score': 0.9542444348335266}]
[{'label': 'Sports', 'score': 0.9640859365463257}]
[{'label': 'World/Politics', 'score': 0.9131503105163574}]
[{'label': 'Business', 'score': 0.8792068958282471}]
